In [15]:
import pandas as pd

In [19]:
annual_crashes = []

for year in range(2019, 2025):
    filename = "accident.csv"
    file_path = f"FARS{year}NationalCSV/{filename}"
    data = pd.read_csv(file_path, encoding="latin1", low_memory=False)
    data.columns = data.columns.str.replace("ï»¿", "", regex=False)
    data["YEAR"] = year
    annual_crashes.append(data)

crashes = pd.concat(annual_crashes, ignore_index=True)

print(crashes["YEAR"].value_counts().sort_index())
print("Total crashes", len(crashes))

YEAR
2019    33487
2020    35935
2021    39785
2022    39422
2023    37769
2024    36297
Name: count, dtype: int64
Total crashes 222695


In [20]:
#Checking for duplicate crasehs
duplicate_crashes = crashes.duplicated(subset=["YEAR", "ST_CASE"]).sum()
print("Duplicate crashes", duplicate_crashes)

Duplicate crashes 0


In [21]:
columns_to_keep = ["YEAR", "ST_CASE", "MONTH", "DAY", "HOUR", "DAY_WEEK", "STATE", "WEATHER"]
crash_data = crashes[columns_to_keep].copy()

print("Rows and columns", crash_data.shape)
crash_data.head()

Rows and columns (222695, 8)


,YEAR,ST_CASE,MONTH,DAY,HOUR,DAY_WEEK,STATE,WEATHER
0,2019,10001,2,7,12,5,1,1
1,2019,10002,1,23,18,4,1,2
2,2019,10003,1,22,19,3,1,10
3,2019,10004,1,22,3,3,1,1
4,2019,10005,1,18,5,6,1,5


In [22]:
#Looking for missing values
print("Crash hour")
print(crash_data["HOUR"].value_counts().sort_index())

print("\nWeather")
print(crash_data["WEATHER"].value_counts().sort_index())

Crash hour
HOUR
0      9073
1      8088
2      8298
3      6201
4      5676
5      7100
6      7960
7      7047
8      5857
9      5901
10     6663
11     7320
12     8448
13     8896
14     9987
15    10957
16    11042
17    12084
18    13188
19    12764
20    13307
21    13272
22    11642
23    10256
99     1668
Name: count, dtype: int64

Weather
WEATHER
1     161635
2      15395
3        220
4       1809
5       2349
6        357
7         59
8        194
10     29077
11       183
12       183
98     10160
99      1074
Name: count, dtype: int64


In [23]:
missing = float("nan")

crash_data["HOUR"] = crash_data["HOUR"].replace(99, missing)
crash_data["WEATHER"] = crash_data["WEATHER"].replace([98, 99], missing)

print(crash_data[["HOUR", "WEATHER"]].isna().sum())
print("Total crashes", len(crash_data))

HOUR        1668
WEATHER    11234
dtype: int64
Total crashes 222695


In [ ]:
#Calculating the percentage of crashes missing hour or weather
total_crashes = len(crash_data)
missing_counts = crash_data[["HOUR", "WEATHER"]].isna().sum()
missing_percentages = missing_counts / total_crashes * 100

print("Percentage of crashes with missing values")
print(missing_percentages.round(2))

Percentage of crashes with missing values
HOUR       0.75
WEATHER    5.04
dtype: float64


In [ ]:
#Creating a crash date from year, month, and day
crash_data["DATE"] = pd.to_datetime(crash_data[["YEAR", "MONTH", "DAY"]])

print(crash_data[["YEAR", "MONTH", "DAY", "DATE"]].head())

   YEAR  MONTH  DAY       DATE
0  2019      2    7 2019-02-07
1  2019      1   23 2019-01-23
2  2019      1   22 2019-01-22
3  2019      1   22 2019-01-22
4  2019      1   18 2019-01-18


In [28]:
#First day of each holiday window
holiday_starts = pd.DataFrame({
    "YEAR": [2019, 2020, 2021, 2022, 2023, 2024],
    "New Year's Day": [
        "2019-01-01", "2020-01-01", "2021-01-01",
        "2022-01-01", "2023-01-01", "2024-01-01"
    ],
    "Memorial Day": [
        "2019-05-25", "2020-05-23", "2021-05-29",
        "2022-05-28", "2023-05-27", "2024-05-25"
    ],
    "Independence Day": [
        "2019-07-03", "2020-07-03", "2021-07-03",
        "2022-07-03", "2023-07-03", "2024-07-03"
    ],
    "Labor Day": [
        "2019-08-31", "2020-09-05", "2021-09-04",
        "2022-09-03", "2023-09-02", "2024-08-31"
    ],
    "Thanksgiving": [
        "2019-11-28", "2020-11-26", "2021-11-25",
        "2022-11-24", "2023-11-23", "2024-11-28"
    ],
    "Christmas": [
        "2019-12-24", "2020-12-24", "2021-12-24",
        "2022-12-24", "2023-12-24", "2024-12-24"
    ]
})
print(holiday_starts)

   YEAR New Year's Day Memorial Day Independence Day   Labor Day Thanksgiving  \
0  2019     2019-01-01   2019-05-25       2019-07-03  2019-08-31   2019-11-28   
1  2020     2020-01-01   2020-05-23       2020-07-03  2020-09-05   2020-11-26   
2  2021     2021-01-01   2021-05-29       2021-07-03  2021-09-04   2021-11-25   
3  2022     2022-01-01   2022-05-28       2022-07-03  2022-09-03   2022-11-24   
4  2023     2023-01-01   2023-05-27       2023-07-03  2023-09-02   2023-11-23   
5  2024     2024-01-01   2024-05-25       2024-07-03  2024-08-31   2024-11-28   

    Christmas  
0  2019-12-24  
1  2020-12-24  
2  2021-12-24  
3  2022-12-24  
4  2023-12-24  
5  2024-12-24  
